# TradeSense · Component 03 · notebook 50: have we seen a situation like this before? Historical contextual intelligence (protocol H1 v3; CPU, minutes)

**Question.** For the current hour of BTC or ETH, can the engine find genuinely similar *past* hours of the same pair and show what happened after them, without peeking into the future and without returning near-duplicates?

**What it does.** Builds a versioned index of 1-hour windows (18 numbers each: fused and model score summaries + the strongest normalised feature deviations), retrieves the 5 most similar past windows (one per distinct episode, at least 2 hours apart), and tests: (H-H1) does it find abnormal / same-family injected events, (H-H2) do the neighbours say something about the next hour on genuine Sep/Oct bars, (H-H3) no duplicates, no future leakage, fast enough, (H-H4) stable under small noise. Baselines: random windows, score-only, feature-only, and a plain persistence rule. **Retrieval is context, not a forecast and not a signal.**

No model is retrained or changed. TEST-B was opened once in T0, so the Sep/Oct results are a **new pre-registered analysis on saved scores**; the protocol text below is hashed when this notebook runs.

**Protocol text hashed by this notebook:**


```
# Protocol H1 v3 (go-ahead given by the user 2026-10-16; frozen when notebook 50 hashes this exact text) - historical contextual intelligence (novelty 4)
Written BEFORE any retrieval number exists. Nothing frozen (models, 6 features, normalisation, fusion stacker, thresholds q) changes. CPU, no training. TEST-3 / TEST-B were opened once in T0: TEST results here are a NEW pre-registered analysis on saved scores (genuine TEST-B bars and their saved scores); nothing is retuned afterwards. Library sha 3b5d9df23521adf9.
Change log v2 -> v3 (made before any number existed): H-H1 is made stricter and split into H-H1a / H-H1b with explicit precision thresholds instead of "3x chance", because chance depends on how many event windows the archive holds; the archive holds ONE labelled event window per injected event.

## Idea
For the current 1-hour window of a trading pair, find the most similar PAST windows of the same pair and show what happened after them (score trajectory, how long the elevated state lasted, whether Alert was reached, realised volatility). Descriptive context only: not price direction, not a signal, never "confirmed manipulation".

## Method
- Window = 60 causal bars ending at t. Descriptor (18 numbers): mean, max and last value of the fused score and of each model score (AE A6, XGBoost, Transformer) = 12; maximum over the window of |z| of each of the 6 N1-normalised features = 6. Standardised per symbol with the mean and standard deviation of the genuine DEV windows (fitted once, frozen). A window is VALID only if all 60 bars lie in one continuous segment, all 18 numbers are finite and the segment has at least 1,500 bars of history; the 60 bars after it (outcome) must also exist in the same segment for archive membership.
- Timeline per symbol: DEV segment (Layer 1 bars 90,720..129,599, July 2024) followed by Aug-Oct 2024 (132,480 bars). Windows never span the seam. Scores: DEV = out-of-fold fused score and model scores; Aug-Oct = saved scores of notebook 32 (fusion and models).
- Retrieval: same pair, Euclidean distance on the standardised descriptor, top 5, PAST ONLY: neighbour end + 60 outcome bars < query window start. Returned neighbours must end >= 120 bars apart (one match per distinct episode), so overlapping or near-duplicate windows are never returned. Fewer than 5 valid matches => fewer are returned, never invalid ones.
- Index version = sha256 of (descriptor specification + standardiser + list of archive window end times), saved with the index.
- Shown per match (dashboard, later): timestamp; distance as "closer than X% of all archive windows"; descriptor items and model scores; Integrity State of the window RECONSTRUCTED by applying the frozen declared Watch / Alert lines (rolling 7-day genuine fused quantiles, causal) and labelled "reconstructed"; what happened in the next 60 bars: peak fused score, minutes above the Alert line, Alert reached yes/no, realised volatility (std of log returns). Fixed sentence: similar past patterns do not guarantee the same future.

## Evaluation and hypotheses (reported as failed if they fail)
Baselines: b0 random past valid windows; b1 score-only descriptor (12 numbers); b2 feature-only descriptor (6 numbers); b3 the full descriptor (proposed).
- Evaluation archive for H-H1 only: genuine DEV windows + genuine Aug windows (past-only) + ONE labelled injected-event window per VAL-3 event (window ending at the event's last bar, both replicas), family = label. Queries: one window per CAL-3 event (replica 0) ending at the event's last bar.
- H-H1a (retrieval finds abnormal-looking history): precision@5 of "neighbour is an injected event window" >= 0.50 (b3, mean of BTC and ETH).
- H-H1b (retrieval finds the same kind of pattern): precision@5 of "neighbour is an event window of the same family" >= 0.25 (b3), and no baseline b0, b1, b2 exceeds b3 by more than 0.02.
- H-H2 (history informs what follows): archive = genuine windows only (product setting). Queries = genuine TEST-B windows ending on every full hour from the 1,440th bar of Sep 1. Spearman between the mean of the 5 neighbours' next-hour peak fused score and the actual next-hour peak, mean over BTC and ETH, >= 0.20 with day-block bootstrap 95% interval (2,000 draws, seed 0) above 0, and b3 >= b0 + 0.10.
- H-H3 (duplicates, causality, latency): no query ever returns two neighbours < 120 bars apart; no neighbour or its outcome bars overlap or follow the query (unit test and full check on all queries); median query time over the full genuine archive <= 50 ms.
- H-H4 (robustness): adding Gaussian noise with standard deviation 0.05 (standardised units) to the query descriptor keeps >= 60% of the top-5 neighbours (mean overlap, H-H2 queries).
- Descriptive (no decision): persistence baseline (Spearman between the current window's maximum fused score and the next-hour peak), P@5 by family, share of genuine queries whose neighbours were mostly Alert windows, example match tables.

## Stated in advance
Short history (July + Aug-Oct 2024, two symbols): "similar past" can be thin, and the 2024 drift of trade counts is only partly removed by N1 normalisation. Labels are synthetic. The descriptor is hand-designed, not learned. Outcomes are descriptive score and volatility behaviour; retrieval is context, not a forecaster (a persistence baseline is reported to show this). Cross-market and text retrieval belong to the teammate's Retrieval-Augmented engine (boundary to be confirmed by the user).

```

In [ ]:
import os, sys, json, time, hashlib, warnings, glob
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'protocol B2 missing'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
PROTO_H1 = r"""# Protocol H1 v3 (go-ahead given by the user 2026-10-16; frozen when notebook 50 hashes this exact text) - historical contextual intelligence (novelty 4)
Written BEFORE any retrieval number exists. Nothing frozen (models, 6 features, normalisation, fusion stacker, thresholds q) changes. CPU, no training. TEST-3 / TEST-B were opened once in T0: TEST results here are a NEW pre-registered analysis on saved scores (genuine TEST-B bars and their saved scores); nothing is retuned afterwards. Library sha 3b5d9df23521adf9.
Change log v2 -> v3 (made before any number existed): H-H1 is made stricter and split into H-H1a / H-H1b with explicit precision thresholds instead of "3x chance", because chance depends on how many event windows the archive holds; the archive holds ONE labelled event window per injected event.

## Idea
For the current 1-hour window of a trading pair, find the most similar PAST windows of the same pair and show what happened after them (score trajectory, how long the elevated state lasted, whether Alert was reached, realised volatility). Descriptive context only: not price direction, not a signal, never "confirmed manipulation".

## Method
- Window = 60 causal bars ending at t. Descriptor (18 numbers): mean, max and last value of the fused score and of each model score (AE A6, XGBoost, Transformer) = 12; maximum over the window of |z| of each of the 6 N1-normalised features = 6. Standardised per symbol with the mean and standard deviation of the genuine DEV windows (fitted once, frozen). A window is VALID only if all 60 bars lie in one continuous segment, all 18 numbers are finite and the segment has at least 1,500 bars of history; the 60 bars after it (outcome) must also exist in the same segment for archive membership.
- Timeline per symbol: DEV segment (Layer 1 bars 90,720..129,599, July 2024) followed by Aug-Oct 2024 (132,480 bars). Windows never span the seam. Scores: DEV = out-of-fold fused score and model scores; Aug-Oct = saved scores of notebook 32 (fusion and models).
- Retrieval: same pair, Euclidean distance on the standardised descriptor, top 5, PAST ONLY: neighbour end + 60 outcome bars < query window start. Returned neighbours must end >= 120 bars apart (one match per distinct episode), so overlapping or near-duplicate windows are never returned. Fewer than 5 valid matches => fewer are returned, never invalid ones.
- Index version = sha256 of (descriptor specification + standardiser + list of archive window end times), saved with the index.
- Shown per match (dashboard, later): timestamp; distance as "closer than X% of all archive windows"; descriptor items and model scores; Integrity State of the window RECONSTRUCTED by applying the frozen declared Watch / Alert lines (rolling 7-day genuine fused quantiles, causal) and labelled "reconstructed"; what happened in the next 60 bars: peak fused score, minutes above the Alert line, Alert reached yes/no, realised volatility (std of log returns). Fixed sentence: similar past patterns do not guarantee the same future.

## Evaluation and hypotheses (reported as failed if they fail)
Baselines: b0 random past valid windows; b1 score-only descriptor (12 numbers); b2 feature-only descriptor (6 numbers); b3 the full descriptor (proposed).
- Evaluation archive for H-H1 only: genuine DEV windows + genuine Aug windows (past-only) + ONE labelled injected-event window per VAL-3 event (window ending at the event's last bar, both replicas), family = label. Queries: one window per CAL-3 event (replica 0) ending at the event's last bar.
- H-H1a (retrieval finds abnormal-looking history): precision@5 of "neighbour is an injected event window" >= 0.50 (b3, mean of BTC and ETH).
- H-H1b (retrieval finds the same kind of pattern): precision@5 of "neighbour is an event window of the same family" >= 0.25 (b3), and no baseline b0, b1, b2 exceeds b3 by more than 0.02.
- H-H2 (history informs what follows): archive = genuine windows only (product setting). Queries = genuine TEST-B windows ending on every full hour from the 1,440th bar of Sep 1. Spearman between the mean of the 5 neighbours' next-hour peak fused score and the actual next-hour peak, mean over BTC and ETH, >= 0.20 with day-block bootstrap 95% interval (2,000 draws, seed 0) above 0, and b3 >= b0 + 0.10.
- H-H3 (duplicates, causality, latency): no query ever returns two neighbours < 120 bars apart; no neighbour or its outcome bars overlap or follow the query (unit test and full check on all queries); median query time over the full genuine archive <= 50 ms.
- H-H4 (robustness): adding Gaussian noise with standard deviation 0.05 (standardised units) to the query descriptor keeps >= 60% of the top-5 neighbours (mean overlap, H-H2 queries).
- Descriptive (no decision): persistence baseline (Spearman between the current window's maximum fused score and the next-hour peak), P@5 by family, share of genuine queries whose neighbours were mostly Alert windows, example match tables.

## Stated in advance
Short history (July + Aug-Oct 2024, two symbols): "similar past" can be thin, and the 2024 drift of trade counts is only partly removed by N1 normalisation. Labels are synthetic. The descriptor is hand-designed, not learned. Outcomes are descriptive score and volatility behaviour; retrieval is context, not a forecaster (a persistence baseline is reported to show this). Cross-market and text retrieval belong to the teammate's Retrieval-Augmented engine (boundary to be confirmed by the user).
"""
PHH = hashlib.sha256(PROTO_H1.encode()).hexdigest(); pp = OUT/'protocol_h1_v3.md'
if pp.exists(): assert pp.read_text() == PROTO_H1, 'protocol H1 v3 exists and differs from this notebook'
else: pp.write_text(PROTO_H1)
(OUT/'protocol_h1_v3.sha256').write_text(PHH + '  protocol_h1_v3.md\n')
for f in ['test_b_results_v1.json', 'declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_fusion_v0.json', 'declared_watch_v1.json', 'b2_lib.py', 'c03_infer.py']: assert (OUT/f).exists(), f'{f} missing'
T32 = json.loads((OUT/'test_b_results_v1.json').read_text())
for s in SYMBOLS: assert sha_file(OUT/'scores'/f'test_b_{s}.npz') == T32['test_score_files_sha16'][s], 'saved TEST scores differ from what notebook 32 recorded'
assert sha_file(OUT/'c03_infer.py') == T32['library_sha16'] and sha_file(OUT/'b2_lib.py') == T32['b2_lib_sha16'], 'library files differ from the ones that opened TEST'
sys.path.insert(0, str(OUT)); import b2_lib as B; import c03_infer as c
say('protocol H1 v3 sha256 ' + PHH[:16] + '; protocol B2 intact ' + ph[:16] + '; library ' + T32['library_sha16'] + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol H1 v3 sha256 e47ee068038c057f; protocol B2 intact 443f99d5faf5929f; library 3b5d9df23521adf9


## 1 · Definitions (descriptor, outcome, past-only retrieval)

In [ ]:
W = 60; OUTW = 60; GAP = 120; K = 5; WARM = B.WARM; AOLEN = 31 * 1440; AOFULL = 132480; TEST_LO = AOLEN; DEV_LO, DEV_HI = 90720, 129600; DEVN = DEV_HI - DEV_LO; HIST_MIN = 1500
SERIES = ['fused', 'AE', 'XGBoost', 'Transformer']; DNAMES = [f'{s}_{st}' for s in SERIES for st in ('mean', 'max', 'last')] + [f'maxabs_z_{f}' for f in B.FEATS]; assert len(DNAMES) == 18
SPEC = {'window_bars': W, 'outcome_bars': OUTW, 'gap_bars': GAP, 'k': K, 'dims': DNAMES, 'metric': 'euclidean on per-symbol standardised descriptor', 'history_min_bars': HIST_MIN}
def raw_desc(F4, zabs, hist_min=HIST_MIN):
    """(n, 18) window descriptors for windows ending at each bar (NaN where invalid). F4 = 4 score arrays; zabs = (n, 6) |z|."""
    n = len(F4[0]); cols = []
    for x in F4:
        s = pd.Series(np.asarray(x, 'float64')); r = s.rolling(W, min_periods=W); cols += [r.mean().to_numpy(), r.max().to_numpy(), s.to_numpy()]
    zm = pd.DataFrame(np.asarray(zabs, 'float64')).rolling(W, min_periods=W).max().to_numpy(); D_ = np.column_stack(cols + [zm[:, j] for j in range(6)])
    D_[np.arange(n) < hist_min + W - 1] = np.nan; return D_
def forward_outcomes(F, thrA, logret):
    """for a window ending at e: next OUTW bars e+1..e+OUTW -> peak fused, minutes above Alert line, Alert reached, realised vol. NaN if the bars do not all exist."""
    n = len(F); pk = np.full(n, np.nan); ma = np.full(n, np.nan); vo = np.full(n, np.nan)
    if n > OUTW:
        from numpy.lib.stride_tricks import sliding_window_view as swv
        Fw = swv(F, OUTW)[1:]; Aw = swv((F > thrA).astype(float), OUTW)[1:]; Lw = swv(logret, OUTW)[1:]; m = len(Fw)
        pk[:m] = Fw.max(axis=1); ma[:m] = Aw.sum(axis=1); vo[:m] = Lw.std(axis=1)
    return pk, ma, vo
def select(dist, times, k=K, gap=GAP):
    """indices of up to k nearest candidates, nearest first, every pair >= gap bars apart"""
    order = np.argsort(dist, kind='stable'); picked = []
    for i in order:
        if all(abs(times[i] - times[j]) >= gap for j in picked): picked.append(i)
        if len(picked) == k: break
    return picked
def retrieve(A, At, q, q_end, k=K, gap=GAP):
    """A: standardised archive (n, d) sorted by end time At; q: query descriptor; past-only: archive end + OUTW < query start (q_end - W + 1)"""
    n = int(np.searchsorted(At, q_end - W + 1 - OUTW, side='left'))      # At < q_start - OUTW
    if n == 0: return np.array([], int), np.array([])
    d = np.sqrt(((A[:n] - q) ** 2).sum(axis=1)); m = min(n, 600); part = np.argpartition(d, m - 1)[:m] if n > m else np.arange(n)
    sel = select(d[part], At[part], k, gap)
    if len(sel) < k and n > m: sel = select(d, At[:n], k, gap); return np.array(sel, int), d[sel]
    ii = part[sel]; return np.array(ii, int), d[ii]

## 2 · Genuine timeline (July DEV + Aug-Oct), scores, outcomes

In [ ]:
DF = json.load(open(OUT/'declared_final_set_v1.json')); FUSD = json.load(open(OUT/'declared_fusion_v0.json')); DW = json.load(open(OUT/'declared_watch_v1.json'))
MODELS = c.MODEL_NAMES; FNORM = {m: DF['models'][m]['normalisation'] for m in MODELS}; FILES = {'AE-ctx fmax': 'ae_ctx_agg', 'XGBoost v2': 'xgb_v2', 'Transformer v2': 'transformer_v2'}; TAGM = dict(zip(MODELS, ['AE', 'XGB', 'TR']))
QA = FUSD['q_T1_by_budget']['2']; QW = DW['watch']['q']['4']
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:WARM] = np.nan; return a
def find1(pat):
    cc = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert cc, pat; return sorted(cc)[0]
def fix(d): d = d.copy(); d.loc[d.index[:15], 'volume_acceleration'] = 0.0; return d
def mcol(zz, k): x = zz[k]; return x[:, 0] if x.ndim == 2 else x
G = {}      # per symbol: genuine timeline (DEV then Aug-Oct) and injected series for the H-H1 evaluation
for s in SYMBOLS:
    a = fix(pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet'))); a = a.set_index('datetime') if 'datetime' in a.columns else a
    b = fix(pd.read_parquet(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet')); assert len(b) == AOFULL, len(b)
    N0 = B.fit_n0(a[B.FEATS].reset_index(drop=True).iloc[15:90720])
    zL1 = np.abs(B.normalise(a[B.FEATS].reset_index(drop=True), 'N1', N0)); zAO = np.abs(B.normalise(b[B.FEATS].reset_index(drop=True), 'N1', N0))
    zf = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); tz = np.load(OUT/'scores'/f'test_b_{s}.npz'); assert 'FUS__GEN' in tz.files
    dev = [nanw(zf['OOF_GEN_L1'])[DEV_LO:DEV_HI]]; ao = [nanw(tz['FUS__GEN'])]
    for m in MODELS:
        zz = np.load(OUT/'scores'/f'{FILES[m]}_{s}_{FNORM[m]}.npz'); dev.append(nanw(mcol(zz, 'GEN_L1'))[DEV_LO:DEV_HI]); ao.append(nanw(tz[TAGM[m] + '__GEN']))
    # parity: the August part of the saved TEST file equals the fusion August scores of notebook 35
    dmax = np.nanmax(np.abs(ao[0][:AOLEN] - nanw(zf['GEN_AO'])[:AOLEN])); say(f'{s}: parity of August fused scores (TEST file vs notebook 35): max difference {dmax:.2e}')
    ts = np.concatenate([a.index.to_numpy()[DEV_LO:DEV_HI], b.index.to_numpy()]).astype('datetime64[ns]')
    Fg = np.concatenate([dev[0], ao[0]]); thrA = c.rolling_threshold(Fg, QA[s]); thrW = c.rolling_threshold(Fg, QW[s])
    seg = np.concatenate([np.zeros(DEVN, int), np.ones(AOFULL, int)]); lr = np.concatenate([a['log_returns'].to_numpy()[DEV_LO:DEV_HI], b['log_returns'].to_numpy()])
    RD = np.vstack([raw_desc([x for x in [dev[0]] + dev[1:]], zL1[DEV_LO:DEV_HI], hist_min=0), raw_desc(ao[0:1] + ao[1:], zAO, hist_min=HIST_MIN)])
    pk, ma, vo = forward_outcomes(Fg[:DEVN], thrA[:DEVN], lr[:DEVN]); pk2, ma2, vo2 = forward_outcomes(Fg[DEVN:], thrA[DEVN:], lr[DEVN:]); PK, MA, VO = np.concatenate([pk, pk2]), np.concatenate([ma, ma2]), np.concatenate([vo, vo2])
    st = np.where(Fg > thrA, 2, np.where(Fg > thrW, 1, 0)); stp = pd.Series(st).rolling(W, min_periods=1).max().to_numpy().astype(int)
    G[s] = dict(ts=ts, F=Fg, thrA=thrA, thrW=thrW, seg=seg, lr=lr, RD=RD, PK=PK, MA=MA, VO=VO, st=st, stp=stp, N0=N0, parts=(dev, ao), zL1=zL1, zAO=zAO, a=a, b=b)
    say(f'{s}: genuine timeline {len(Fg)} bars (DEV {DEVN} + Aug-Oct {AOFULL}); valid windows with outcome {int((np.isfinite(RD).all(axis=1) & np.isfinite(PK)).sum())}')

BTCUSDT: parity of August fused scores (TEST file vs notebook 35): max difference 9.61e-03
BTCUSDT: genuine timeline 171360 bars (DEV 38880 + Aug-Oct 132480); valid windows with outcome 169622
ETHUSDT: parity of August fused scores (TEST file vs notebook 35): max difference 7.44e-03
ETHUSDT: genuine timeline 171360 bars (DEV 38880 + Aug-Oct 132480); valid windows with outcome 169622


## 3 · Versioned index

In [ ]:
# ---- standardiser from genuine DEV windows only; archive = genuine valid windows with an outcome
STD = {}; ARCH = {}
def seam_ok(seg_arr):
    ok = np.ones(len(seg_arr), bool); ok[DEVN:DEVN + W - 1] = ok[DEVN:DEVN + W - 1]; return ok
for s in SYMBOLS:
    g = G[s]; valid = np.isfinite(g['RD']).all(axis=1) & np.isfinite(g['PK']) & np.isfinite(g['MA']) & np.isfinite(g['VO'])
    devm = valid & (np.arange(len(valid)) < DEVN); mu = g['RD'][devm].mean(axis=0); sd = g['RD'][devm].std(axis=0) + 1e-9; STD[s] = (mu, sd)
    Z = (g['RD'] - mu) / sd; idx = np.flatnonzero(valid)
    ARCH[s] = dict(Z=Z[idx].astype('float64'), t=idx.astype(np.int64), PK=g['PK'][idx], MA=g['MA'][idx], VO=g['VO'][idx], st_last=g['st'][idx], st_peak=g['stp'][idx], lab=np.zeros(len(idx), int), Zall=Z)
    say(f'{s}: standardiser from {int(devm.sum())} genuine DEV windows; genuine archive {len(idx)} windows ({idx.min()}..{idx.max()})')
def index_version(s):
    h = hashlib.sha256(); h.update(json.dumps(SPEC, sort_keys=True).encode()); h.update(STD[s][0].tobytes()); h.update(STD[s][1].tobytes()); h.update(ARCH[s]['t'].tobytes()); return h.hexdigest()
VERS = {s: index_version(s) for s in SYMBOLS}
for s in SYMBOLS:
    A = ARCH[s]; np.savez_compressed(OUT/f'history_index_v1_{s}.npz', Z=A['Z'].astype('float32'), t=A['t'], PK=A['PK'], MA=A['MA'], VO=A['VO'], st_last=A['st_last'], st_peak=A['st_peak'], ts=G[s]['ts'][A['t']].astype('datetime64[ns]').astype('int64'), mean=STD[s][0], std=STD[s][1])
(OUT/'history_index_v1.json').write_text(json.dumps({'spec': SPEC, 'version_sha256': VERS, 'protocol_h1_v3_sha256': PHH, 'archive_windows': {s: int(len(ARCH[s]['t'])) for s in SYMBOLS}, 'note': 'index files are data: kept on Drive, never committed'}, indent=1))
say('index versions: ' + str({s: VERS[s][:16] for s in SYMBOLS}))

BTCUSDT: standardiser from 38761 genuine DEV windows; genuine archive 169622 windows (59..171299)
ETHUSDT: standardiser from 38761 genuine DEV windows; genuine archive 169622 windows (59..171299)
index versions: {'BTCUSDT': '43bb821cb1a8a6ad', 'ETHUSDT': '553deb2848300fd0'}


## 4 · Unit tests

In [ ]:
# T01 descriptor equals a hand computation
rg = np.random.default_rng(0); n = 3000; F4 = [rg.normal(size=n).cumsum() * 0.01 + i for i in range(4)]; zz = np.abs(rg.normal(size=(n, 6))); Dd = raw_desc(F4, zz, hist_min=0); e = 2000; w = F4[1][e - W + 1:e + 1]
assert abs(Dd[e, 3] - w.mean()) < 1e-12 and abs(Dd[e, 4] - w.max()) < 1e-12 and abs(Dd[e, 5] - F4[1][e]) < 1e-12 and np.allclose(Dd[e, 12:], zz[e - W + 1:e + 1].max(axis=0)) and np.isnan(Dd[W - 2]).all() and np.isfinite(Dd[W - 1]).all()
say('PASS T01 window descriptor (mean / max / last of each score, max |z| of each feature) equals a hand calculation; the first 59 bars are invalid')
# T02 causality: changing every value after bar t leaves the descriptor at t unchanged
F4b = [np.where(np.arange(n) > e, x * 5 + 3, x) for x in F4]; zb = np.where(np.arange(n)[:, None] > e, zz * 4, zz); Db = raw_desc(F4b, zb, hist_min=0); assert np.allclose(Dd[:e + 1], Db[:e + 1], equal_nan=True)
say('PASS T02 the descriptor at bar t does not depend on later bars')
# T03 retrieval equals a plain brute-force reference, is past-only and separated
At = np.sort(rg.integers(0, 50000, 4000)); At = np.unique(At); A_ = rg.normal(size=(len(At), 18))
for q_end in [20000, 33333, 49999]:
    q = rg.normal(size=18); ii, dd = retrieve(A_, At, q, q_end)
    cand = np.flatnonzero(At + OUTW < q_end - W + 1); dist = np.sqrt(((A_[cand] - q) ** 2).sum(axis=1)); ref = []
    for j in np.argsort(dist, kind='stable'):
        if all(abs(At[cand[j]] - At[r]) >= GAP for r in ref): ref.append(cand[j])
        if len(ref) == K: break
    assert list(ii) == [int(x) for x in ref] and (At[ii] + OUTW < q_end - W + 1).all() and all(abs(At[i] - At[j]) >= GAP for a_, i in enumerate(ii) for j in ii[:a_])
ii, dd = retrieve(A_, At, rg.normal(size=18), 10); assert len(ii) == 0
say('PASS T03 retrieval equals a brute-force reference; every neighbour and its outcome bars end before the query window starts; neighbours are >= 120 bars apart; an empty past returns nothing')
# T04 outcome uses strictly later bars only
Fo = np.arange(300.0); pk, ma, vo = forward_outcomes(Fo, np.full(300, 150.0), np.zeros(300)); assert pk[10] == 70.0 and ma[100] == 0.0 + np.sum(np.arange(101, 161) > 150) and np.isnan(pk[250]) and not np.isnan(pk[239])
say('PASS T04 outcome of a window = the next 60 bars only (the last 60 bars of a segment have no outcome)')
# T05 windows never span the DEV / Aug seam; standardiser comes from genuine DEV windows only
for s in SYMBOLS:
    assert np.isnan(G[s]['RD'][DEVN:DEVN + HIST_MIN + W - 1]).all() and np.isfinite(G[s]['RD'][DEVN - 1000:DEVN]).all(axis=1).any()
    t_ = ARCH[s]['t']; assert not ((t_ >= DEVN - OUTW) & (t_ < DEVN + HIST_MIN + W - 1)).any()
    dm = np.isfinite(G[s]['RD'][:DEVN]).all(axis=1) & np.isfinite(G[s]['PK'][:DEVN]); assert np.allclose(STD[s][0], G[s]['RD'][:DEVN][dm].mean(axis=0))
say('PASS T05 no archive window spans the seam or lacks history; the standardiser uses genuine DEV windows only')
# T06 the index version is deterministic
assert VERS == {s: index_version(s) for s in SYMBOLS}; say('PASS T06 the index version hash is deterministic')

PASS T01 window descriptor (mean / max / last of each score, max |z| of each feature) equals a hand calculation; the first 59 bars are invalid
PASS T02 the descriptor at bar t does not depend on later bars
PASS T03 retrieval equals a brute-force reference; every neighbour and its outcome bars end before the query window starts; neighbours are >= 120 bars apart; an empty past returns nothing
PASS T04 outcome of a window = the next 60 bars only (the last 60 bars of a segment have no outcome)
PASS T05 no archive window spans the seam or lacks history; the standardiser uses genuine DEV windows only
PASS T06 the index version hash is deterministic


## 5 · H-H1: injected events (CAL-3 queries)

In [ ]:
# ---------------- H-H1: injected events (CAL-3 queries; archive = genuine past + one labelled VAL-3 event window per event)
EVA = {}; QRY = {}
def zinj(frame, N0): return np.abs(B.normalise(frame[B.FEATS].reset_index(drop=True), 'N1', N0))
for s in SYMBOLS:
    g = G[s]; zf = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); mu, sd = STD[s]
    # VAL-3 injected windows (labels), DEV segment only
    vp = pd.read_parquet(L3/f'l3v3_{s}_VAL.parquet'); ve = pd.read_csv(L3/f'l3v3_{s}_VAL_events.csv'); Zs, Ts, Ls = [], [], []
    for r in sorted(vp.replica.unique()):
        f = vp[vp.replica == r].reset_index(drop=True); zz = zinj(f, g['N0']); sc = [nanw(zf[f'OOF_VAL_r{r}'])]
        for m in MODELS: sc.append(nanw(mcol(np.load(OUT/'scores'/f'{FILES[m]}_{s}_{FNORM[m]}.npz'), f'VAL_r{r}')))
        Rv = raw_desc([x[DEV_LO:DEV_HI] for x in sc], zz[DEV_LO:DEV_HI], hist_min=0); lab = f.label.to_numpy().astype(int)
        for ev in ve[ve.replica == r].itertuples():
            e = ev.t0 + ev.dur - 1 - DEV_LO
            if e < 0 or e >= DEVN or not np.isfinite(Rv[e]).all(): continue
            wl = set(np.unique(lab[ev.t0 + ev.dur - W:ev.t0 + ev.dur])) - {0}
            if wl and wl != {int(ev.family)}: continue
            Zs.append((Rv[e] - mu) / sd); Ts.append(e); Ls.append(int(ev.family))
    # CAL-3 queries
    cp = pd.read_parquet(L3/f'l3v3_{s}_CAL.parquet'); cp = cp[cp.replica == 0].reset_index(drop=True).iloc[:AOLEN]; ce = pd.read_csv(L3/f'l3v3_{s}_CAL_events.csv'); ce = ce[ce.replica == 0]
    zc = zinj(cp, g['N0']); scc = [nanw(zf['CAL_r0'][:AOLEN])] + [nanw(mcol(np.load(OUT/'scores'/f'{FILES[m]}_{s}_{FNORM[m]}.npz'), 'CAL_r0')[:AOLEN]) for m in MODELS]
    Rc = raw_desc(scc, zc, hist_min=HIST_MIN); qs = []
    for ev in ce.itertuples():
        e = ev.t0 + ev.dur - 1
        if e < AOLEN and np.isfinite(Rc[e]).all(): qs.append((DEVN + e, (Rc[e] - mu) / sd, int(ev.family)))
    A = ARCH[s]; Zc = np.vstack([A['Z']] + ([np.array(Zs)] if Zs else [])); Tc = np.concatenate([A['t'], np.array(Ts, np.int64)]); Lc = np.concatenate([A['lab'], np.array(Ls, int)]); o = np.argsort(Tc, kind='stable')
    EVA[s] = dict(Z=Zc[o], t=Tc[o], lab=Lc[o]); QRY[s] = qs
    say(f'{s}: H-H1 archive {len(Tc)} windows (genuine {len(A["t"])}, labelled VAL-3 event windows {len(Ts)}); CAL-3 query windows {len(qs)}')
def sub(Z, cols): return Z[:, cols]
VARS = {'b3 full (proposed)': np.arange(0, 18), 'b1 score-only': np.arange(0, 12), 'b2 feature-only': np.arange(12, 18)}
def p_at5(s, name, cols, rng):
    E = EVA[s]; Zs_ = E['Z'][:, cols]; out = []
    for (qe, qd, fam) in QRY[s]:
        if name == 'b0 random past':
            n = int(np.searchsorted(E['t'], qe - W + 1 - OUTW, side='left')); pool = rng.permutation(n); sel = select(np.zeros(n)[pool], E['t'][:n][pool], K, GAP); ii = pool[sel]
        else: ii, _ = retrieve(Zs_, E['t'], qd[cols], qe)
        if len(ii) == 0: continue
        l_ = E['lab'][ii]; out.append((fam, float((l_ > 0).mean()), float((l_ == fam).mean()), len(ii)))
    return pd.DataFrame(out, columns=['fam', 'abn', 'same', 'n'])
HH1 = {}; rows = []; FAMP = {}
for name, cols in list(VARS.items()) + [('b0 random past', np.arange(0, 18))]:
    per = []
    for s in SYMBOLS:
        r_ = p_at5(s, name, cols, np.random.default_rng(1)); per.append(r_); rows.append({'variant': name, 'sym': s, 'queries': len(r_), 'P@5 any event window': r_.abn.mean(), 'P@5 same family': r_.same.mean(), 'mean neighbours returned': r_.n.mean()})
    HH1[name] = per
    if name.startswith('b3'): FAMP = pd.concat(per).groupby('fam')[['abn', 'same']].mean()
H1T = pd.DataFrame(rows); H1T.to_csv(OUT/'h1_hh1_table.csv', index=False)
mean_ = lambda name, col: float(np.mean([r_[col].mean() for r_ in HH1[name]]))
HH1a = mean_('b3 full (proposed)', 'abn'); HH1b = mean_('b3 full (proposed)', 'same'); best_other = max(mean_(n, 'same') for n in ['b0 random past', 'b1 score-only', 'b2 feature-only'])
HC1a = bool(HH1a >= 0.50); HC1b = bool(HH1b >= 0.25 and best_other <= HH1b + 0.02)
FAMP.index = [B.FAMS[int(i)] for i in FAMP.index]
say('--- H-H1 · retrieval of injected events (CAL-3 query windows; archive = genuine past + labelled VAL-3 events) ---'); say(H1T.round(3).to_string(index=False)); say(''); say('b3 by family (mean of both symbols):'); say(FAMP.round(3).to_string())

BTCUSDT: H-H1 archive 169917 windows (genuine 169622, labelled VAL-3 event windows 295); CAL-3 query windows 200
ETHUSDT: H-H1 archive 169918 windows (genuine 169622, labelled VAL-3 event windows 296); CAL-3 query windows 200
--- H-H1 · retrieval of injected events (CAL-3 query windows; archive = genuine past + labelled VAL-3 events) ---
           variant     sym  queries  P@5 any event window  P@5 same family  mean neighbours returned
b3 full (proposed) BTCUSDT      200                 0.564            0.270                       5.0
b3 full (proposed) ETHUSDT      200                 0.594            0.224                       5.0
     b1 score-only BTCUSDT      200                 0.557            0.260                       5.0
     b1 score-only ETHUSDT      200                 0.612            0.224                       5.0
   b2 feature-only BTCUSDT      200                 0.100            0.039                       5.0
   b2 feature-only ETHUSDT      200                 0.

## 6 · H-H2 to H-H4: genuine TEST-B hourly queries

In [ ]:
# ---------------- H-H2/H-H3/H-H4: genuine TEST-B hourly queries, product archive (genuine windows only)
def day_boot(vals_a, vals_b, days, n=NBOOT, seed=0):
    ud = np.unique(days); rg = np.random.default_rng(seed); ix = {d: np.flatnonzero(days == d) for d in ud}; out = []
    for _ in range(n):
        pick = rg.choice(ud, len(ud)); ii = np.concatenate([ix[d] for d in pick]); out.append(spearmanr(vals_a[ii], vals_b[ii])[0])
    return np.percentile(out, [2.5, 97.5])
R2 = {}; LAT = []; DUP = 0; CAUS = 0; NQ = 0; OVL = []
for s in SYMBOLS:
    g = G[s]; A = ARCH[s]; qe = np.arange(DEVN + TEST_LO + 1440, DEVN + AOFULL - OUTW - 1); qe = qe[(qe - DEVN) % 60 == 0]; qe = qe[np.isfinite(A['Zall'][qe]).all(axis=1) & np.isfinite(g['PK'][qe])]
    days = (qe - DEVN - TEST_LO) // 1440; rg = np.random.default_rng(3); act = g['PK'][qe]; pred = {k: [] for k in ['b3 full (proposed)', 'b1 score-only', 'b2 feature-only', 'b0 random past']}; persist = g['RD'][qe, 1]; ex = []
    mostly = []
    for j, e in enumerate(qe):
        q = A['Zall'][e]
        for name, cols in list(VARS.items()):
            t0_ = time.perf_counter(); ii, dd = retrieve(A['Z'][:, cols], A['t'], q[cols], e); lat = time.perf_counter() - t0_
            if name.startswith('b3'):
                LAT.append(lat); NQ += 1; T_ = A['t'][ii]
                CAUS += int((T_ + OUTW >= e - W + 1).sum()); DUP += int(sum(abs(T_[a_] - T_[b_]) < GAP for a_ in range(len(T_)) for b_ in range(a_)))
                mostly.append(bool((A['st_peak'][ii] == 2).sum() >= 3))
                if len(ii) and j % 25 == 0:
                    qn = q[cols] + np.random.default_rng(j).normal(0, 0.05, len(cols)); i2, _ = retrieve(A['Z'][:, cols], A['t'], qn, e); OVL.append(len(set(ii.tolist()) & set(i2.tolist())) / max(len(ii), 1))
                if len(ex) < 400 and j % 40 == 0: ex.append((e, ii.copy(), dd.copy()))
            pred[name].append(A['PK'][ii].mean() if len(ii) else np.nan)
        n_ = int(np.searchsorted(A['t'], e - W + 1 - OUTW, side='left')); pool = rg.permutation(n_)[:200]; sel = select(np.zeros(len(pool)), A['t'][pool], K, GAP); pred['b0 random past'].append(A['PK'][pool[sel]].mean())
    P = {k: np.array(v) for k, v in pred.items()}; ok = np.all([np.isfinite(v) for v in P.values()], axis=0) & np.isfinite(act)
    R2[s] = dict(q=qe[ok], days=days[ok], act=act[ok], P={k: v[ok] for k, v in P.items()}, persist=persist[ok], mostly=np.array(mostly)[ok], ex=ex)
    say(f'{s}: H-H2 queries {int(ok.sum())} (hourly genuine TEST-B windows)')
SP = {name: {s: float(spearmanr(R2[s]['P'][name], R2[s]['act'])[0]) for s in SYMBOLS} for name in R2[SYMBOLS[0]]['P']}; SP['persistence (current window max)'] = {s: float(spearmanr(R2[s]['persist'], R2[s]['act'])[0]) for s in SYMBOLS}
mean_sp = {k: float(np.mean(list(v.values()))) for k, v in SP.items()}
cat = lambda k: np.concatenate([R2[s]['P'][k] for s in SYMBOLS]) if k in R2[SYMBOLS[0]]['P'] else None
def boot_mean(name, n=NBOOT):
    ud = {s: np.unique(R2[s]['days']) for s in SYMBOLS}; rg = np.random.default_rng(0); out = []
    for _ in range(n):
        vals = []
        for s in SYMBOLS:
            d_ = rg.choice(ud[s], len(ud[s])); ii = np.concatenate([np.flatnonzero(R2[s]['days'] == d) for d in d_]); vals.append(spearmanr(R2[s]['P'][name][ii], R2[s]['act'][ii])[0])
        out.append(np.mean(vals))
    return np.percentile(out, [2.5, 97.5])
ci2 = boot_mean('b3 full (proposed)'); HH2 = bool(mean_sp['b3 full (proposed)'] >= 0.20 and ci2[0] > 0 and mean_sp['b3 full (proposed)'] >= mean_sp['b0 random past'] + 0.10)
SPT = pd.DataFrame([{'predictor of next-hour peak fused score': k, 'Spearman BTC': v['BTCUSDT'], 'Spearman ETH': v['ETHUSDT'], 'mean': mean_sp[k]} for k, v in SP.items()]); SPT.to_csv(OUT/'h1_hh2_table.csv', index=False)
say(''); say('--- H-H2 · do the 5 neighbours say something about the next hour? (genuine TEST-B hourly windows) ---'); say(SPT.round(3).to_string(index=False)); say(f'b3 mean Spearman 95% day-block interval {ci2[0]:.3f}..{ci2[1]:.3f}')
lat_ms = float(np.median(LAT) * 1000); HH3 = bool(DUP == 0 and CAUS == 0 and lat_ms <= 50); HH4v = float(np.mean(OVL)) if OVL else np.nan; HH4 = bool(HH4v >= 0.60)
fc = float(np.mean(np.concatenate([R2[s]['mostly'] for s in SYMBOLS])))
say(f'H-H3 queries {NQ}: neighbours closer than 120 bars: {DUP}; neighbour/outcome overlapping or following the query: {CAUS}; median query time {lat_ms:.1f} ms (archive {np.mean([len(ARCH[s]["t"]) for s in SYMBOLS]):.0f} windows)')
say(f'H-H4 mean overlap of the top-5 under noise sd 0.05: {HH4v:.3f}; descriptive: share of genuine queries whose neighbours were mostly Alert windows (>= 3 of 5): {fc:.3f}')

BTCUSDT: H-H2 queries 1439 (hourly genuine TEST-B windows)
ETHUSDT: H-H2 queries 1439 (hourly genuine TEST-B windows)

--- H-H2 · do the 5 neighbours say something about the next hour? (genuine TEST-B hourly windows) ---
predictor of next-hour peak fused score  Spearman BTC  Spearman ETH   mean
                     b3 full (proposed)         0.059         0.112  0.086
                          b1 score-only         0.068         0.138  0.103
                        b2 feature-only        -0.050         0.024 -0.013
                         b0 random past         0.034         0.014  0.024
       persistence (current window max)         0.033         0.142  0.087
b3 mean Spearman 95% day-block interval 0.048..0.121
H-H3 queries 2878: neighbours closer than 120 bars: 0; neighbour/outcome overlapping or following the query: 0; median query time 34.9 ms (archive 169622 windows)
H-H4 mean overlap of the top-5 under noise sd 0.05: 0.822; descriptive: share of genuine queries whose neighbours

## 7 · Example match table

In [ ]:
# ---------------- example match tables (what the dashboard will show); clearly from genuine TEST-B bars
def show(s, e, ii, dd):
    g = G[s]; A = ARCH[s]; n_ = int(np.searchsorted(A['t'], e - W + 1 - OUTW, side='left')); dall = np.sqrt(((A['Z'][:n_] - A['Zall'][e]) ** 2).sum(axis=1)); names = {0: 'NORMAL', 1: 'WATCH', 2: 'ALERT'}
    rows = [{'match': i + 1, 'window ended (UTC)': str(g['ts'][A['t'][j]])[:16], 'closer than % of past windows': round(100 * float((dall > d).mean()), 1), 'state (reconstructed)': names[int(A['st_last'][j])], 'peak fused next hour': round(float(A['PK'][j]), 2), 'minutes above Alert next hour': int(A['MA'][j]), 'volatility next hour': float(f"{A['VO'][j]:.2e}")} for i, (j, d) in enumerate(zip(ii, dd))]
    return pd.DataFrame(rows)
s = SYMBOLS[0]; ex = R2[s]['ex']
if ex:
    e, ii, dd = sorted(ex, key=lambda x: -G[s]['PK'][x[0]] if np.isfinite(G[s]['PK'][x[0]]) else 0)[0]
    say(''); say(f'--- example: {s} window ended {str(G[s]["ts"][e])[:16]} UTC (actual next-hour peak {G[s]["PK"][e]:.2f}); 5 most similar PAST windows ---'); say(show(s, e, ii, dd).to_string(index=False)); say('Similar past patterns do not guarantee the same future. States are reconstructed with the frozen Watch / Alert lines. Not a price forecast.')


--- example: BTCUSDT window ended 2024-10-07T00:00 UTC (actual next-hour peak 3.06); 5 most similar PAST windows ---
 match window ended (UTC)  closer than % of past windows state (reconstructed)  peak fused next hour  minutes above Alert next hour  volatility next hour
     1   2024-09-17T09:31                          100.0                NORMAL                  1.07                              0              0.000494
     2   2024-09-02T19:07                          100.0                NORMAL                 -0.18                              0              0.000399
     3   2024-10-06T20:01                          100.0                NORMAL                  0.11                              0              0.000302
     4   2024-09-08T07:26                          100.0                NORMAL                  0.14                              0              0.000277
     5   2024-09-03T02:04                          100.0                NORMAL                 -0.46            

## 8 · Results (protocol H1 v3)

In [ ]:
say(''); say('=== pre-registered hypotheses (protocol H1 v3; H-H2 to H-H4 are a new analysis on saved TEST-B scores) ===')
say(f'H-H1a precision@5 "neighbour is an injected event window" >= 0.50: {HH1a:.3f} -> {"HOLDS" if HC1a else "FAILS"}')
say(f'H-H1b precision@5 "same family" >= 0.25 and no baseline better by more than 0.02: b3 {HH1b:.3f}, best other {best_other:.3f} -> {"HOLDS" if HC1b else "FAILS"}')
say(f'H-H2 Spearman(neighbour mean peak, actual next-hour peak) >= 0.20 with interval above 0 and >= random + 0.10: {mean_sp["b3 full (proposed)"]:.3f} (interval {ci2[0]:.3f}..{ci2[1]:.3f}; random {mean_sp["b0 random past"]:.3f}) -> {"HOLDS" if HH2 else "FAILS"}')
say(f'H-H3 no duplicates, no future leakage, median query <= 50 ms: duplicates {DUP}, leakage {CAUS}, {lat_ms:.1f} ms -> {"HOLDS" if HH3 else "FAILS"}')
say(f'H-H4 top-5 overlap under noise >= 0.60: {HH4v:.3f} -> {"HOLDS" if HH4 else "FAILS"}')
HYP = {'H-H1a': HC1a, 'H-H1b': HC1b, 'H-H2': HH2, 'H-H3': HH3, 'H-H4': HH4}; say(f'{sum(HYP.values())} of 5 hold. Failed hypotheses are reported as failed; nothing was retuned.')
say(f'Descriptive: persistence (current window max fused score) Spearman {mean_sp["persistence (current window max)"]:.3f} versus retrieval {mean_sp["b3 full (proposed)"]:.3f}. Retrieval is historical CONTEXT, not a forecaster. Labels are synthetic; history is short (July + Aug-Oct 2024).')
RES = {'protocol_h1_v3_sha256': PHH, 'index_versions': VERS, 'hypotheses': HYP, 'H-H1a': HH1a, 'H-H1b': HH1b, 'best_other_same_family': best_other, 'spearman': mean_sp, 'spearman_ci_b3': [float(x) for x in ci2], 'duplicates': DUP, 'leakage': CAUS, 'median_query_ms': lat_ms, 'noise_overlap': HH4v, 'share_mostly_alert_neighbours': fc, 'mock': MOCK}


=== pre-registered hypotheses (protocol H1 v3; H-H2 to H-H4 are a new analysis on saved TEST-B scores) ===
H-H1a precision@5 "neighbour is an injected event window" >= 0.50: 0.579 -> HOLDS
H-H1b precision@5 "same family" >= 0.25 and no baseline better by more than 0.02: b3 0.247, best other 0.242 -> FAILS
H-H2 Spearman(neighbour mean peak, actual next-hour peak) >= 0.20 with interval above 0 and >= random + 0.10: 0.086 (interval 0.048..0.121; random 0.024) -> FAILS
H-H3 no duplicates, no future leakage, median query <= 50 ms: duplicates 0, leakage 0, 34.9 ms -> HOLDS
H-H4 top-5 overlap under noise >= 0.60: 0.822 -> HOLDS
3 of 5 hold. Failed hypotheses are reported as failed; nothing was retuned.
Descriptive: persistence (current window max fused score) Spearman 0.087 versus retrieval 0.086. Retrieval is historical CONTEXT, not a forecaster. Labels are synthetic; history is short (July + Aug-Oct 2024).


## 9 · Save

In [ ]:
(OUT/'h1_results.json').write_text(json.dumps(RES, indent=1, default=str)); (OUT/'h1_share_block.txt').write_text('\n'.join(LINES))
say('Saved: h1_results.json, h1_hh1_table.csv, h1_hh2_table.csv, history_index_v1_*.npz, history_index_v1.json, h1_share_block.txt in models_b2_v1/ . Send the share block to Claude.')

Saved: h1_results.json, h1_hh1_table.csv, h1_hh2_table.csv, history_index_v1_*.npz, history_index_v1.json, h1_share_block.txt in models_b2_v1/ . Send the share block to Claude.
